In [ ]:
import pickle
import numpy as np
import pandas as pd

!pip install bertopic
!pip install zstandard
from bertopic import BERTopic
from sklearn.feature_extraction.text import CountVectorizer
from zstd import *



In [ ]:
def save_pkl(tgt_list, svg_path):
    with open(svg_path, "wb") as f:
        pickle.dump(tgt_list, f)

def load_pkl(path) :
    with open(path, "rb") as f:
        data = pickle.load(f)
    return data

preprocessing

In [ ]:
submission = zstd_reader('data/college/college_submissions.zst')
comment = zstd_reader('data/college/college_comments.zst')

submission_df = unpack_submssion(submission)
comment_df = unpack_comment(comment)

In [ ]:
def process_submission_df(df):
  processed_title = df[(df['title'] != '[deleted]') & (df['title'] != '[removed]') &
                                       (df['title'] != '[deleted by user]') & (df['title'] != '')]


  processed_body = df[(df['selftext'] != '[deleted]') & (df['selftext'] != '[removed]') &
                                       (df['selftext'] != '[deleted by user]') & (df['selftext'] != '')]

  processed_df = pd.concat([processed_title, processed_body])

  processed_df = processed_df.drop_duplicates()
  processed_df = processed_df.reset_index(drop=True)
  processed_df['combined'] = processed_df[['title', 'selftext']].agg('\n'.join, axis=1)
  processed_df['combined'] = processed_df['combined'].map(lambda x: x[:-10] if x[-1] == ']' else x)

  processed_df['year'] = processed_df['post_created_utc'].dt.year
  processed_df['month'] = processed_df['post_created_utc'].dt.month
  processed_df['timestamp'] = processed_df.apply(lambda x: f"{x['year']}-{x['month']:02d}", axis=1)

  processed_df_pre = processed_df[processed_df.post_created_utc < '2020-01-01'].reset_index(drop=True)
  processed_df_post = processed_df[processed_df.post_created_utc >= '2020-01-01'].reset_index(drop=True)

  return processed_df, processed_df_pre, processed_df_post

def process_comment_df(df):

  processed_df = df[(df['body'] != '[deleted]') & (df['body'] != '[removed]') &
                                       (df['body'] != '[deleted by user]') & (df['body'] != '')]
  processed_df = processed_df.reset_index(drop=True)

  return processed_df

processed_submission_df, df_pre, df_post = process_submission_df(submission_df)
processed_comment_df = process_comment_df(comment_df)
print(submission_df.shape)
print(processed_submission_df.shape)
print(df_pre.shape)
print(df_post.shape)
processed_submission_df.head(2)

In [ ]:
print(df_pre.shape)
print(df_post.shape)
print('Number of unique authors:', len(set(processed_submission_df['post_author'])))
print('Number of unique authors before COVID:', len(set(df_pre['post_author'])))
print('Number of unique authors after COVID:', len(set(df_post['post_author'])))

In [ ]:
from sklearn.feature_extraction import text

# Customize stop words
additional = ['thanks', 'thank', 'thank you', 'com', 'https', 'reddit', 'subreddit', 'f1', 'visa']
stop_words = list(text.ENGLISH_STOP_WORDS.union(additional))

In [ ]:
# define docs (pre-covid)
docs = list(df_pre['combined'])

# define docs (post-covid)
# docs = list(df_post['combined'])

In [ ]:
vectorizer_model = CountVectorizer(ngram_range = (1,3), stop_words = stop_words)
x = vectorizer_model.fit_transform(docs)
vectorizer_model.get_feature_names_out()

In [ ]:
from sklearn.cluster import KMeans
from sentence_transformers import SentenceTransformer
from bertopic.vectorizers import ClassTfidfTransformer

# remove stopwords
vectorizer_model = CountVectorizer(ngram_range = (1,3), stop_words = stop_words)
model_embedding = SentenceTransformer('all-MiniLM-L6-v2')
ctfidf_model = ClassTfidfTransformer(reduce_frequent_words=True)
corpus_embeddings = model_embedding.encode(docs)

In [ ]:
embedding_column = pd.DataFrame({'embeddings':[corpus_embeddings[i,:] for i in range(corpus_embeddings.shape[0])]})
df_topic = pd.concat([df_pre, embedding_column], axis=1)
print(df_topic.shape)
df_topic.head(2)

# save_pkl(df, 'data/submission_with_embeddings.pkl')

In [ ]:
cluster_model = KMeans(n_clusters = 7)
model = BERTopic(
    embedding_model='xlm-r-bert-base-nli-stsb-mean-tokens',
    language = 'english',
    hdbscan_model = cluster_model,
    vectorizer_model = vectorizer_model,
    ctfidf_model = ctfidf_model,
    calculate_probabilities=True,
    verbose = True,
    nr_topics = 7
)

# # # dynamic topic modeling
# timestamps = df.period.to_list()
# topics_over_time = model.topics_over_time(docs, timestamps, nr_bins=20)

In [ ]:
topics, probs = model.fit_transform(docs)

In [ ]:
freq = model.get_topic_info()
freq

In [ ]:
print(set(topics))
df_topic['Topic'] = topics
df_topic.head(3)

In [ ]:
# import smart_open
import gensim.corpora as corpora
from gensim.models.coherencemodel import CoherenceModel

documents = pd.DataFrame({"Document": docs,
                          "ID": range(len(docs)),
                          "Topic": topics})
documents_per_topic = documents.groupby(['Topic'], as_index=False).agg({'Document': ' '.join})
cleaned_docs = model._preprocess_text(documents_per_topic.Document.values)

# Extract vectorizer and analyzer from BERTopic
vectorizer = model.vectorizer_model
analyzer = vectorizer.build_analyzer()

# Extract features for Topic Coherence evaluation
words = vectorizer.get_feature_names_out()
tokens = [analyzer(doc) for doc in cleaned_docs]
dictionary = corpora.Dictionary(tokens)
corpus = [dictionary.doc2bow(token) for token in tokens]
topic_words = [[words for words, _ in model.get_topic(topic)]
               for topic in range(len(set(topics))-1)]

# Evaluate
coherence_model = CoherenceModel(topics = topic_words,
                                 texts = tokens,
                                 corpus = corpus,
                                 dictionary = dictionary,
                                 coherence = 'c_v'
                                 )
coherence = coherence_model.get_coherence()

print(coherence)

* Coherence Score (CV)

Topic = 3 : 0.8835126843051759

Topic = 4 : 0.8338150824575377

Topic = 5 : 0.8447945902993583

Topic = 6 : 0.8574816803746682

Topic = 7 : 0.7153931796502565

Topic = 8 : 0.7936147353426992

Topic = 9 : 0.7877565224317387

Topic = 10 : 0.7781596653610514